In [ ]:
# ==========================================
# IMPROVED HYBRID XLM-RoBERTa + TF-IDF + STAT
# WITH COMPREHENSIVE ABLATION STUDY
# ==========================================
import os, re, warnings, math, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from transformers import (
    XLMRobertaTokenizer, XLMRobertaModel,
    get_cosine_schedule_with_warmup
)
from torch.optim import AdamW

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import json

# ---------------------
# 0. Utils & Reproducibility
# ---------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------
# 1. Load datasets
# ---------------------
train_df = pd.read_excel("train_data_with_codemixed_preprocessing.xlsx")
val_df   = pd.read_excel("val_data_with_codemixed_preprocessing.xlsx")
test_df  = pd.read_excel("test_data_with_codemixed_preprocessing.xlsx")

# ---------------------
# 2. Ablation Configuration Class
# ---------------------
class AblationConfig:
    def __init__(self, 
                 use_tfidf_word=True,
                 use_tfidf_char=True,
                 use_stat_features=True,
                 use_attention=True,
                 use_rdrop=True,
                 use_focal_loss=True,
                 use_ema=True,
                 use_llrd=True,
                 use_residual=True,
                 use_layer_norm=True,
                 unfreeze_last_n_layers=4,
                 name="baseline"):
        self.use_tfidf_word = use_tfidf_word
        self.use_tfidf_char = use_tfidf_char
        self.use_stat_features = use_stat_features
        self.use_attention = use_attention
        self.use_rdrop = use_rdrop
        self.use_focal_loss = use_focal_loss
        self.use_ema = use_ema
        self.use_llrd = use_llrd
        self.use_residual = use_residual
        self.use_layer_norm = use_layer_norm
        self.unfreeze_last_n_layers = unfreeze_last_n_layers
        self.name = name
    
    def to_dict(self):
        return {
            'use_tfidf_word': self.use_tfidf_word,
            'use_tfidf_char': self.use_tfidf_char,
            'use_stat_features': self.use_stat_features,
            'use_attention': self.use_attention,
            'use_rdrop': self.use_rdrop,
            'use_focal_loss': self.use_focal_loss,
            'use_ema': self.use_ema,
            'use_llrd': self.use_llrd,
            'use_residual': self.use_residual,
            'use_layer_norm': self.use_layer_norm,
            'unfreeze_last_n_layers': self.unfreeze_last_n_layers,
            'name': self.name
        }

# ---------------------
# 3. Generate Ablation Configurations
# ---------------------
def generate_ablation_configs():
    configs = []
    
    # Baseline: All components enabled
    configs.append(AblationConfig(name="baseline_full"))
    
    # Single component ablations (remove one at a time)
    configs.append(AblationConfig(use_tfidf_word=False, name="no_tfidf_word"))
    configs.append(AblationConfig(use_tfidf_char=False, name="no_tfidf_char"))
    configs.append(AblationConfig(use_stat_features=False, name="no_stat_features"))
    configs.append(AblationConfig(use_attention=False, name="no_attention"))
    configs.append(AblationConfig(use_rdrop=False, name="no_rdrop"))
    configs.append(AblationConfig(use_focal_loss=False, name="no_focal_loss"))
    configs.append(AblationConfig(use_ema=False, name="no_ema"))
    configs.append(AblationConfig(use_llrd=False, name="no_llrd"))
    configs.append(AblationConfig(use_residual=False, name="no_residual"))
    configs.append(AblationConfig(use_layer_norm=False, name="no_layer_norm"))
    
    # TF-IDF ablations
    configs.append(AblationConfig(use_tfidf_word=False, use_tfidf_char=False, name="no_tfidf_all"))
    configs.append(AblationConfig(use_tfidf_word=True, use_tfidf_char=False, name="tfidf_word_only"))
    configs.append(AblationConfig(use_tfidf_word=False, use_tfidf_char=True, name="tfidf_char_only"))
    
    # Feature combinations
    configs.append(AblationConfig(use_tfidf_word=False, use_tfidf_char=False, 
                                 use_stat_features=False, name="xlm_only"))
    
    # Training technique ablations
    configs.append(AblationConfig(use_rdrop=False, use_focal_loss=False, 
                                 use_ema=False, name="no_advanced_training"))
    
    # Architecture ablations
    configs.append(AblationConfig(use_attention=False, use_residual=False, 
                                 use_layer_norm=False, name="simple_architecture"))
    
    # Unfreeze layer ablations
    configs.append(AblationConfig(unfreeze_last_n_layers=0, name="all_frozen"))
    configs.append(AblationConfig(unfreeze_last_n_layers=2, name="unfreeze_2_layers"))
    configs.append(AblationConfig(unfreeze_last_n_layers=8, name="unfreeze_8_layers"))
    
    return configs

# ---------------------
# 4. Feature Engineering (Modified for Ablation)
# ---------------------
def prepare_features_ablation(train_df, val_df, test_df, config,
                             max_word_features=1200, max_char_features=600):
    results = {}
    
    # TF-IDF Word features
    if config.use_tfidf_word:
        tfidf_word = TfidfVectorizer(
            max_features=max_word_features,
            ngram_range=(1, 2),
            analyzer='word',
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        train_texts = train_df['tweet_clean_codemixed'].astype(str).tolist()
        val_texts = val_df['tweet_clean_codemixed'].astype(str).tolist()
        test_texts = test_df['tweet_clean_codemixed'].astype(str).tolist()
        
        Xw_tr = tfidf_word.fit_transform(train_texts).toarray()
        Xw_va = tfidf_word.transform(val_texts).toarray()
        Xw_te = tfidf_word.transform(test_texts).toarray()
        results['tfidf_word'] = (Xw_tr, Xw_va, Xw_te, tfidf_word)
    else:
        # Create dummy features
        dummy_dim = max_word_features
        Xw_tr = np.zeros((len(train_df), dummy_dim), dtype='float32')
        Xw_va = np.zeros((len(val_df), dummy_dim), dtype='float32')
        Xw_te = np.zeros((len(test_df), dummy_dim), dtype='float32')
        results['tfidf_word'] = (Xw_tr, Xw_va, Xw_te, None)
    
    # TF-IDF Char features
    if config.use_tfidf_char:
        tfidf_char = TfidfVectorizer(
            max_features=max_char_features,
            ngram_range=(3, 5),
            analyzer='char_wb',
            min_df=2,
            sublinear_tf=True
        )
        train_texts = train_df['tweet_clean_codemixed'].astype(str).tolist()
        val_texts = val_df['tweet_clean_codemixed'].astype(str).tolist()
        test_texts = test_df['tweet_clean_codemixed'].astype(str).tolist()
        
        Xc_tr = tfidf_char.fit_transform(train_texts).toarray()
        Xc_va = tfidf_char.transform(val_texts).toarray()
        Xc_te = tfidf_char.transform(test_texts).toarray()
        results['tfidf_char'] = (Xc_tr, Xc_va, Xc_te, tfidf_char)
    else:
        # Create dummy features
        dummy_dim = max_char_features
        Xc_tr = np.zeros((len(train_df), dummy_dim), dtype='float32')
        Xc_va = np.zeros((len(val_df), dummy_dim), dtype='float32')
        Xc_te = np.zeros((len(test_df), dummy_dim), dtype='float32')
        results['tfidf_char'] = (Xc_tr, Xc_va, Xc_te, None)
    
    # Combine TF-IDF features
    X_tfidf_train = np.hstack([results['tfidf_word'][0], results['tfidf_char'][0]])
    X_tfidf_val = np.hstack([results['tfidf_word'][1], results['tfidf_char'][1]])
    X_tfidf_test = np.hstack([results['tfidf_word'][2], results['tfidf_char'][2]])
    
    # Statistical features
    if config.use_stat_features:
        def extract_enhanced_stat_features(lang_seq, text):
            lang_seq = str(lang_seq)
            text = str(text)
            id_count = len(re.findall(r'\[id\]', lang_seq))
            en_count = len(re.findall(r'\[en\]', lang_seq))
            unk_count = len(re.findall(r'\[\?\?\]', lang_seq))
            total = id_count + en_count + unk_count
            
            words = text.split()
            text_length = len(words)
            avg_word_length = np.mean([len(w) for w in words]) if text_length > 0 else 0.0
            uppercase_ratio = sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0.0
            digit_ratio = sum(1 for c in text if c.isdigit()) / len(text) if len(text) > 0 else 0.0
            
            return [
                float(total),
                float(id_count/total if total else 0.0),
                float(en_count/total if total else 0.0),
                float(unk_count/total if total else 0.0),
                float(text_length),
                float(avg_word_length),
                float(uppercase_ratio),
                float(digit_ratio)
            ]
        
        X_stat_train = np.array([extract_enhanced_stat_features(lang, txt)
                                for lang, txt in zip(train_df['lang_per_word'], train_df['tweet_clean_codemixed'])],
                               dtype='float32')
        X_stat_val = np.array([extract_enhanced_stat_features(lang, txt)
                              for lang, txt in zip(val_df['lang_per_word'], val_df['tweet_clean_codemixed'])],
                             dtype='float32')
        X_stat_test = np.array([extract_enhanced_stat_features(lang, txt)
                               for lang, txt in zip(test_df['lang_per_word'], test_df['tweet_clean_codemixed'])],
                              dtype='float32')
        
        scaler = StandardScaler()
        X_stat_train = scaler.fit_transform(X_stat_train).astype('float32')
        X_stat_val = scaler.transform(X_stat_val).astype('float32')
        X_stat_test = scaler.transform(X_stat_test).astype('float32')
    else:
        # Create dummy statistical features
        X_stat_train = np.zeros((len(train_df), 8), dtype='float32')
        X_stat_val = np.zeros((len(val_df), 8), dtype='float32')
        X_stat_test = np.zeros((len(test_df), 8), dtype='float32')
        scaler = None
    
    tfidf_total_dim = X_tfidf_train.shape[1]
    return X_tfidf_train, X_tfidf_val, X_tfidf_test, X_stat_train, X_stat_val, X_stat_test, tfidf_total_dim, results, scaler

# ---------------------
# 5. Dataset (same as before)
# ---------------------
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

class HybridDataset(Dataset):
    def __init__(self, texts, tfidf, stat, labels, max_length=128):
        self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.tfidf = torch.tensor(tfidf, dtype=torch.float32)
        self.stat = torch.tensor(stat, dtype=torch.float32)
        self.labels = torch.tensor((labels.tolist() if hasattr(labels,'tolist') else labels), dtype=torch.long)
        self.max_length = max_length

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = tokenizer(
            text, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'tfidf': self.tfidf[idx],
            'stat': self.stat[idx],
            'labels': self.labels[idx]
        }

    def __len__(self):
        return len(self.labels)

# ---------------------
# 6. Losses and Utility Functions
# ---------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean', label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
        self.label_smoothing = label_smoothing
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none', label_smoothing=label_smoothing)

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

def rdrop_loss(logits1, logits2, ce_fn, targets, alpha=2.0):
    ce = (ce_fn(logits1, targets) + ce_fn(logits2, targets)) / 2.0
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    kl = F.kl_div(p, q.exp(), reduction='batchmean') + F.kl_div(q, p.exp(), reduction='batchmean')
    return ce + alpha * kl / 2.0

def compute_class_weights(y):
    classes = np.unique(y)
    counts = np.array([(y==c).sum() for c in classes], dtype='float32')
    weights = counts.sum() / (len(classes) * counts)
    w_map = {int(c):w for c,w in zip(classes, weights)}
    return torch.tensor([w_map.get(0,1.0), w_map.get(1,1.0)], dtype=torch.float32).to(device)

# ---------------------
# 7. Modified Model for Ablation
# ---------------------
class AblationHybridModel(nn.Module):
    def __init__(self, config, tfidf_dim,
                 tfidf_hidden_dim=256,
                 stat_hidden_dim=32,
                 classifier_hidden_dim=192,
                 dropout_rate=0.3):
        super().__init__()
        self.config = config
        self.xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        self.xlm.gradient_checkpointing_enable()
        
        # Freeze/unfreeze layers based on config
        for p in self.xlm.parameters():
            p.requires_grad = False
        
        if config.unfreeze_last_n_layers > 0:
            for layer in self.xlm.encoder.layer[-config.unfreeze_last_n_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True

        # TF-IDF tower (always present but might receive zero features)
        self.tfidf_layer1 = nn.Linear(tfidf_dim, tfidf_hidden_dim)
        self.tfidf_layer2 = nn.Linear(tfidf_hidden_dim, tfidf_hidden_dim)
        self.tfidf_do = nn.Dropout(dropout_rate)
        self.tfidf_bn = nn.BatchNorm1d(tfidf_hidden_dim) if config.use_layer_norm else nn.Identity()

        # STAT tower
        self.stat_layer1 = nn.Linear(8, stat_hidden_dim)
        self.stat_layer2 = nn.Linear(stat_hidden_dim, stat_hidden_dim)
        self.stat_do = nn.Dropout(dropout_rate)
        self.stat_bn = nn.BatchNorm1d(stat_hidden_dim) if config.use_layer_norm else nn.Identity()

        total_features = 768 + tfidf_hidden_dim + stat_hidden_dim

        # Attention mechanism
        if config.use_attention:
            self.attn = nn.MultiheadAttention(
                embed_dim=total_features, num_heads=8,
                dropout=dropout_rate, batch_first=True
            )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(total_features, classifier_hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(classifier_hidden_dim) if config.use_layer_norm else nn.Identity(),
            nn.Dropout(dropout_rate),
            nn.Linear(classifier_hidden_dim, classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(classifier_hidden_dim // 2) if config.use_layer_norm else nn.Identity(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(classifier_hidden_dim // 2, 2)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, input_ids, attention_mask, tfidf, stat):
        # XLM-R features
        cls = self.xlm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]

        # TF-IDF path
        t = F.relu(self.tfidf_layer1(tfidf))
        t = self.tfidf_do(t)
        t = self.tfidf_bn(t)
        t2 = self.tfidf_layer2(t)
        if self.config.use_residual and t.shape == t2.shape:
            t = F.relu(t + t2)
        else:
            t = F.relu(t2)

        # STAT path
        s = F.relu(self.stat_layer1(stat))
        s = self.stat_do(s)
        s = self.stat_bn(s)
        s2 = self.stat_layer2(s)
        if self.config.use_residual and s.shape == s2.shape:
            s = F.relu(s + s2)
        else:
            s = F.relu(s2)

        # Combine features
        x = torch.cat([cls, t, s], dim=1)

        # Apply attention if enabled
        if self.config.use_attention:
            x_exp = x.unsqueeze(1)
            x_attn, _ = self.attn(x_exp, x_exp, x_exp)
            x = x_attn.squeeze(1)

        logits = self.classifier(x)
        return logits

# ---------------------
# 8. Optimizer with conditional LLRD
# ---------------------
def get_optimizer_params_ablation(model, config, base_lr, weight_decay=1e-4, lr_decay=0.95):
    params = []
    
    if config.use_llrd and config.unfreeze_last_n_layers > 0:
        # LLRD: Layer-wise learning rate decay
        if hasattr(model.xlm, "encoder"):
            enc_layers = list(model.xlm.encoder.layer)
            n = len(enc_layers)
            for i, layer in enumerate(enc_layers):
                if any(p.requires_grad for p in layer.parameters()):
                    depth_from_top = n - i
                    lr = base_lr * (lr_decay ** depth_from_top)
                    params.append({"params": [p for p in layer.parameters() if p.requires_grad],
                                  "lr": lr, "weight_decay": weight_decay})
    else:
        # Standard learning rate for all transformer layers
        transformer_params = [p for p in model.xlm.parameters() if p.requires_grad]
        if transformer_params:
            params.append({"params": transformer_params, "lr": base_lr, "weight_decay": weight_decay})
    
    # Head parameters (always use base learning rate)
    head_params = []
    for m in [model.tfidf_layer1, model.tfidf_layer2, model.stat_layer1, model.stat_layer2, model.classifier]:
        head_params += list(m.parameters())
    if hasattr(model, 'attn') and model.config.use_attention:
        head_params += list(model.attn.parameters())
    
    params.append({"params": head_params, "lr": base_lr, "weight_decay": weight_decay})
    return params

# ---------------------
# 9. EMA Class
# ---------------------
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.clone()

    def update(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad:
                assert name in self.shadow
                self.shadow[name].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model):
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.shadow:
                self.backup[name] = p.data.clone()
                p.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.backup:
                p.data.copy_(self.backup[name])
        self.backup = {}

# ---------------------
# 10. Training Function for Ablation
# ---------------------
def train_ablation_model(config, train_df, val_df, 
                        max_length=128,
                        max_word_features=1200,
                        max_char_features=600,
                        epochs=15,
                        batch_size=32,
                        learning_rate=2e-5,
                        patience=8):
    
    print(f"\n{'='*50}")
    print(f"Training: {config.name}")
    print(f"{'='*50}")
    
    # Prepare features
    Xtf_tr, Xtf_va, _, Xst_tr, Xst_va, _, tfidf_dim, _, _ = prepare_features_ablation(
        train_df, val_df, val_df, config, max_word_features, max_char_features)
    
    # Create datasets
    train_ds = HybridDataset(train_df['tweet_for_model'], Xtf_tr, Xst_tr, train_df['label'], max_length)
    val_ds = HybridDataset(val_df['tweet_for_model'], Xtf_va, Xst_va, val_df['label'], max_length)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    # Create model
    model = AblationHybridModel(config, tfidf_dim).to(device)
    
    # Optimizer
    optim_params = get_optimizer_params_ablation(model, config, learning_rate)
    optimizer = AdamW(optim_params, lr=learning_rate, weight_decay=1e-4)
    
    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    # Loss function
    class_weights = compute_class_weights(train_df['label'].values)
    if config.use_focal_loss:
        ce_base = FocalLoss(gamma=2.0, weight=class_weights)
    else:
        ce_base = nn.CrossEntropyLoss(weight=class_weights)
    
    # EMA
    ema = EMA(model, decay=0.999) if config.use_ema else None
    
    # AMP
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
    
    best_val_f1 = 0.0
    bad = 0
    
    for epoch in range(epochs):
        # Training
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attn = batch['attention_mask'].to(device, non_blocking=True)
            tfidf = batch['tfidf'].to(device, non_blocking=True)
            stat = batch['stat'].to(device, non_blocking=True)
            targets = batch['labels'].to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                if config.use_rdrop:
                    # R-Drop: two forward passes
                    logits1 = model(input_ids, attn, tfidf, stat)
                    logits2 = model(input_ids, attn, tfidf, stat)
                    loss = rdrop_loss(logits1, logits2, ce_base, targets, alpha=2.0)
                else:
                    # Standard training
                    logits = model(input_ids, attn, tfidf, stat)
                    loss = ce_base(logits, targets)
            
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            if ema:
                ema.update(model)
            
            total_loss += loss.item()
        
        # Validation
        model.eval()
        if ema:
            ema.apply_shadow(model)
        
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attn = batch['attention_mask'].to(device)
                tfidf = batch['tfidf'].to(device)
                stat = batch['stat'].to(device)
                targets = batch['labels'].to(device)
                
                logits = model(input_ids, attn, tfidf, stat)
                preds = logits.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(targets.cpu().numpy())
        
        if ema:
            ema.restore(model)
        
        val_f1 = f1_score(val_targets, val_preds, average='weighted')
        val_acc = accuracy_score(val_targets, val_preds)
        
        print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Val F1={val_f1:.4f}, Val Acc={val_acc:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            bad = 0
            # Save best model
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': config.to_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc
            }, f"ablation_models/{config.name}_best.pt")
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    return best_val_f1, val_acc

# ---------------------
# 11. Evaluation Function
# ---------------------
def evaluate_ablation_model(config, test_df, model_path):
    # Prepare features
    _, _, Xtf_te, _, _, Xst_te, tfidf_dim, _, _ = prepare_features_ablation(
        test_df, test_df, test_df, config)
    
    # Create dataset
    test_ds = HybridDataset(test_df['tweet_for_model'], Xtf_te, Xst_te, test_df['label'])
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
    
    # Load model
    model = AblationHybridModel(config, tfidf_dim).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Evaluate
    model.eval()
    test_preds, test_targets, test_probs = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            tfidf = batch['tfidf'].to(device)
            stat = batch['stat'].to(device)
            targets = batch['labels'].to(device)
            
            logits = model(input_ids, attn, tfidf, stat)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(targets.cpu().numpy())
            test_probs.extend(probs.cpu().numpy())
    
    test_f1 = f1_score(test_targets, test_preds, average='weighted')
    test_acc = accuracy_score(test_targets, test_preds)
    try:
        test_auc = roc_auc_score(test_targets, test_probs)
    except:
        test_auc = 0.0
    
    return test_f1, test_acc, test_auc

# ---------------------
# 12. Main Ablation Study
# ---------------------
def run_ablation_study():
    # Create directory for saving models
    os.makedirs("ablation_models", exist_ok=True)
    
    configs = generate_ablation_configs()
    results = []
    
    print(f"Running ablation study with {len(configs)} configurations...")
    
    for i, config in enumerate(configs):
        print(f"\n[{i+1}/{len(configs)}] Starting: {config.name}")
        
        try:
            # Train model
            val_f1, val_acc = train_ablation_model(config, train_df, val_df)
            
            # Evaluate on test set
            model_path = f"ablation_models/{config.name}_best.pt"
            if os.path.exists(model_path):
                test_f1, test_acc, test_auc = evaluate_ablation_model(config, test_df, model_path)
            else:
                test_f1, test_acc, test_auc = 0.0, 0.0, 0.0
            
            result = {
                'config_name': config.name,
                'val_f1': val_f1,
                'val_acc': val_acc,
                'test_f1': test_f1,
                'test_acc': test_acc,
                'test_auc': test_auc,
                **config.to_dict()
            }
            results.append(result)
            
            print(f"✅ {config.name}: Val F1={val_f1:.4f}, Test F1={test_f1:.4f}")
            
        except Exception as e:
            print(f"❌ Error in {config.name}: {str(e)}")
            result = {
                'config_name': config.name,
                'val_f1': 0.0,
                'val_acc': 0.0,
                'test_f1': 0.0,
                'test_acc': 0.0,
                'test_auc': 0.0,
                'error': str(e),
                **config.to_dict()
            }
            results.append(result)
    
    return results

# ---------------------
# 13. Visualization and Analysis
# ---------------------
def analyze_ablation_results(results):
    df_results = pd.DataFrame(results)
    df_results.to_csv("ablation_study_results.csv", index=False)
    
    # Sort by test F1 score
    df_results = df_results.sort_values('test_f1', ascending=False)
    
    print("\n" + "="*80)
    print("ABLATION STUDY RESULTS")
    print("="*80)
    
    print("\nTop 10 Configurations by Test F1:")
    print(df_results[['config_name', 'val_f1', 'test_f1', 'test_acc', 'test_auc']].head(10).to_string(index=False))
    
    # Find baseline
    baseline_idx = df_results[df_results['config_name'] == 'baseline_full'].index
    if len(baseline_idx) > 0:
        baseline_f1 = df_results.loc[baseline_idx[0], 'test_f1']
        print(f"\nBaseline F1: {baseline_f1:.4f}")
        
        print("\nComponent Impact Analysis (vs Baseline):")
        component_analysis = []
        
        component_configs = [
            ('no_tfidf_word', 'TF-IDF Word Features'),
            ('no_tfidf_char', 'TF-IDF Char Features'),
            ('no_stat_features', 'Statistical Features'),
            ('no_attention', 'Attention Mechanism'),
            ('no_rdrop', 'R-Drop Regularization'),
            ('no_focal_loss', 'Focal Loss'),
            ('no_ema', 'EMA'),
            ('no_llrd', 'LLRD'),
            ('no_residual', 'Residual Connections'),
            ('no_layer_norm', 'Layer Normalization')
        ]
        
        for config_name, component_name in component_configs:
            config_rows = df_results[df_results['config_name'] == config_name]
            if len(config_rows) > 0:
                config_f1 = config_rows.iloc[0]['test_f1']
                impact = baseline_f1 - config_f1
                component_analysis.append({
                    'Component': component_name,
                    'F1_without': config_f1,
                    'Impact': impact,
                    'Impact_pct': (impact / baseline_f1) * 100 if baseline_f1 > 0 else 0
                })
        
        df_impact = pd.DataFrame(component_analysis).sort_values('Impact', ascending=False)
        print(df_impact.to_string(index=False))
    
    # Create visualizations
    plt.figure(figsize=(20, 12))
    
    # 1. Overall results comparison
    plt.subplot(2, 3, 1)
    top_configs = df_results.head(15)
    plt.barh(range(len(top_configs)), top_configs['test_f1'])
    plt.yticks(range(len(top_configs)), top_configs['config_name'], fontsize=8)
    plt.xlabel('Test F1 Score')
    plt.title('Top 15 Configurations by Test F1')
    plt.gca().invert_yaxis()
    
    # 2. Component impact
    if len(component_analysis) > 0:
        plt.subplot(2, 3, 2)
        df_impact_sorted = df_impact.sort_values('Impact')
        colors = ['red' if x < 0 else 'green' for x in df_impact_sorted['Impact']]
        plt.barh(range(len(df_impact_sorted)), df_impact_sorted['Impact'], color=colors)
        plt.yticks(range(len(df_impact_sorted)), df_impact_sorted['Component'], fontsize=8)
        plt.xlabel('F1 Impact (Baseline - Without Component)')
        plt.title('Component Impact Analysis')
        plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    
    # 3. Val vs Test F1 correlation
    plt.subplot(2, 3, 3)
    plt.scatter(df_results['val_f1'], df_results['test_f1'], alpha=0.7)
    plt.xlabel('Validation F1')
    plt.ylabel('Test F1')
    plt.title('Validation vs Test F1 Correlation')
    plt.plot([0, 1], [0, 1], 'r--', alpha=0.5)
    
    # 4. Feature ablation comparison
    plt.subplot(2, 3, 4)
    feature_configs = df_results[df_results['config_name'].isin([
        'baseline_full', 'no_tfidf_all', 'tfidf_word_only', 'tfidf_char_only', 'xlm_only'
    ])]
    if len(feature_configs) > 0:
        plt.bar(feature_configs['config_name'], feature_configs['test_f1'])
        plt.xticks(rotation=45)
        plt.ylabel('Test F1')
        plt.title('Feature Combination Analysis')
    
    # 5. Training technique comparison
    plt.subplot(2, 3, 5)
    training_configs = df_results[df_results['config_name'].isin([
        'baseline_full', 'no_rdrop', 'no_focal_loss', 'no_ema', 'no_advanced_training'
    ])]
    if len(training_configs) > 0:
        plt.bar(training_configs['config_name'], training_configs['test_f1'])
        plt.xticks(rotation=45)
        plt.ylabel('Test F1')
        plt.title('Training Technique Analysis')
    
    # 6. Unfreeze layer analysis
    plt.subplot(2, 3, 6)
    layer_configs = df_results[df_results['config_name'].isin([
        'all_frozen', 'unfreeze_2_layers', 'baseline_full', 'unfreeze_8_layers'
    ])]
    if len(layer_configs) > 0:
        layer_nums = [0, 2, 4, 8]  # corresponding to configs
        plt.plot(layer_nums, layer_configs['test_f1'], 'o-')
        plt.xlabel('Number of Unfrozen Layers')
        plt.ylabel('Test F1')
        plt.title('Unfrozen Layers Impact')
    
    plt.tight_layout()
    plt.savefig('ablation_study_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return df_results, df_impact if len(component_analysis) > 0 else None

# ---------------------
# 14. Run the Complete Ablation Study
# ---------------------
if __name__ == "__main__":
    print("🚀 Starting Comprehensive Ablation Study...")
    
    # Run ablation study
    results = run_ablation_study()
    
    # Analyze results
    df_results, df_impact = analyze_ablation_results(results)
    
    # Save detailed analysis
    with open('ablation_analysis.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print("\n🎉 Ablation study completed!")
    print("📊 Results saved to: ablation_study_results.csv")
    print("📈 Visualizations saved to: ablation_study_analysis.png")
    print("📋 Detailed analysis saved to: ablation_analysis.json")
    
    # Print summary
    if len(results) > 0:
        best_config = max(results, key=lambda x: x.get('test_f1', 0))
        print(f"\n🏆 Best Configuration: {best_config['config_name']}")
        print(f"   Test F1: {best_config['test_f1']:.4f}")
        print(f"   Test Accuracy: {best_config['test_acc']:.4f}")
        print(f"   Test AUC: {best_config['test_auc']:.4f}")


Using device: cuda
🚀 Starting Comprehensive Ablation Study...
Running ablation study with 20 configurations...

[1/20] Starting: baseline_full

Training: baseline_full


Epoch 1/15:   0%|                                                                              | 0/274 [00:00<?, ?it/s]